## 회귀 모델(알고리즘)
- 회귀 : 연속적인 수치(숫자)를 예측하는 모델
- 분류 vs 회귀
  - 분류 : 범주(up/down)를 예측하며, 예측이 맞았는지/틀렸는지가 명확함 (혼동행렬, 정확도 등)
  - 회귀 : 연속적인 수치(수익률)를 예측하며, 정확히 맞았다는 개념이 없음! -> 오차의 크기를 측정(MAE, RMSE, R^2 등)
> **회귀에서는 얼마나 가까운가(오차의 크기)**가 가장 중요!\
> 참고: **로지스틱 회귀**는 이름만 회귀이고, 선형 계산 후 확률을 통해 범주를 판단하는 **분류 모델**임!

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _ml import build_dataset, time_split, FEATURES, SEED
from _style import setup

setup()
pd.set_option("display.width", 130)

df = build_dataset()
train, test, cutoff = time_split(df)


tr = train.sample(15000, random_state=SEED)
te = test.sample(4000, random_state=SEED)


X_train, X_test = tr[FEATURES], te[FEATURES]

# 정답(y)을 두 종류 준비 -> 분류(target_up) / 회귀(target_ret)
y_train_cls, y_test_cls = tr["target_up"], te["target_up"]   # 분류용 정답 : 상승(1) / 하락(0)
y_train_reg, y_test_reg = tr["target_ret"], te["target_ret"] # 회귀용 정답 : 수익률(수치)


print(f" * 학습 {len(tr):,}행 ({tr['date'].min().date()} ~ {tr['date'].max().date()})")
print(f" * 테스트 {len(te):,}행 ({te['date'].min().date()} ~ {te['date'].max().date()})")
print(f" * 피처 {len(FEATURES)}개")
print()
print(f"- 분류용 정답(target_up) : {y_train_cls.unique()} (범주)")
print(f"- 회귀용 정답(target_ret) : {y_train_reg.min():.4f} ~ {y_train_reg.max():.4f} (수치)")

 * 학습 15,000행 (2023-10-24 ~ 2026-01-16)
 * 테스트 4,000행 (2026-01-19 ~ 2026-09-07)
 * 피처 7개

- 분류용 정답(target_up) : [0 1] (범주)
- 회귀용 정답(target_ret) : -0.2260 ~ 0.2313 (수치)


- 분류 : y = target_up (0/1) --> RandomForestClassifier
- 회귀 : y = target_ret (수익률) --> RandomForestRegressor

**이름은 동일한 알고리즘이지만, 평가 방법이 완전히 달라짐!**
- RandomForestClassifier => accuracy_score(정확도), f1_score(f1 점수)
- RandomForestRegressor => mean_absolute_error(MAE), mean_squared_error(MSE), r2_score(R^2)

In [3]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (accuracy_score, f1_score,                          # 분류 평가지표
                             mean_absolute_error, mean_squared_error, r2_score) # 회귀 평가지표

clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED, n_jobs=-1)
reg = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=SEED, n_jobs=-1)

# 학습 --> fit(X_train,y_train)
# 분류 모델 학습
clf.fit(X_train, y_train_cls)

# 회귀 모델 학습
reg.fit(X_train, y_train_reg)

# 예측 --> pred = predict(X_test)
# 분류 모델 예측 (pred_cls)
pred_cls = clf.predict(X_test)

# 회귀 모델 예측 (pred_reg)
pred_reg = reg.predict(X_test)


print(f"{'':<14}{'분류':<32}{'회귀'}")
print('-' * 70)
print(f"{'정답예시':<13}{y_test_cls.iloc[0]:<32}{y_test_reg.iloc[0]:.4f}")
print(f"{'예측예시':<13}{pred_cls[0]:<32}{pred_reg[0]:.4f}")
print()
print(f"분류 평가 : 정확도 {accuracy_score(y_test_cls, pred_cls):.4f}"
      f"     F1 {f1_score(y_test_cls, pred_cls):.4f}")

print(f"회귀 평가 : RMSE {np.sqrt(mean_squared_error(y_test_reg, pred_reg)):.6f}"
      f"    R^2 {r2_score(y_test_reg, pred_reg):.4f}")

              분류                              회귀
----------------------------------------------------------------------
정답예시         0                               -0.0068
예측예시         1                               -0.0000

분류 평가 : 정확도 0.4933     F1 0.4269
회귀 평가 : RMSE 0.027389    R^2 -0.0068


In [4]:
# 회귀 예측을 0 기준으로 나누어서 분류를 해본다면?
reg_to_cls = (pred_reg > 0).astype(int)

print(f"회귀 모델 예측을 0 기준으로 나눈 정확도: "
      f" {accuracy_score(y_test_cls, reg_to_cls):.4f}")
print(f"분류 모델로 학습한 정확도: "
      f" {accuracy_score(y_test_cls, pred_cls):.4f}")

회귀 모델 예측을 0 기준으로 나눈 정확도:  0.4793
분류 모델로 학습한 정확도:  0.4933
